In [ ]:
pip install transformers tensorflow sentencepiece torch

In [2]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import pickle
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Input, Embedding, Dense, Concatenate
import copy
from numpy import unique
from sklearn.preprocessing import LabelEncoder
from keras.models import Model
from keras.layers import Input
from keras.layers import Dense
from keras.layers import Embedding
from keras.layers import concatenate
from keras.utils import plot_model
import math
from tensorflow.keras.layers import Reshape
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import LSTM
from transformers import XLMRobertaTokenizer, TFAutoModel
from transformers import TFAutoModelForSequenceClassification
import matplotlib.pyplot as plt
#from keras.utils import plot_model
import os
import torch
from huggingface_hub import from_pretrained_keras
from collections import Counter
import tensorflow as tf
from tensorflow.keras import optimizers
from tensorflow.keras import metrics
from tensorflow.keras import losses
from transformers import pipeline
from keras.layers import Softmax
from tensorflow.keras.utils import plot_model
from tensorflow.keras.optimizers.schedules import PolynomialDecay
from tensorflow.keras import regularizers
from keras import backend as K
from sklearn.metrics import mean_absolute_error

In [3]:
#pandas settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [5]:
#let's filter on the outliers and see how model performance increases
def filter_outliers(df, upper, lower, target_col = "award_contract/val_total: 0"):
    """This function only works on numerical columns"""
    data_array = np.sort(df[target_col].to_numpy())
    cut_off_low_indice = math.floor(lower * len(data_array))
    cut_off_high_indice = math.floor(upper * len(data_array)) - 1
    low_amount = data_array[cut_off_low_indice]
    high_amount = data_array[cut_off_high_indice]

    outlier_indices = []

    for i in range(0, len(df)):
        if df[target_col].iloc[i] > high_amount:
            outlier_indices.append(i)
        elif df[target_col].iloc[i] < low_amount:
            outlier_indices.append(i)
        else:
            continue

    print("{} rows were dropped because of outliers".format(len(outlier_indices)))
    df = df.drop(labels = outlier_indices)

In [6]:
def merge_text_cols(df):
  text_columns = ["short_descr", "ac criteria", "object_contract/title", "object descr titles", "ac cost criteria"]
  merged_text = []
  for i in range(0, len(df)):
    text_row = ""
    for col in text_columns:
      text_cell = str(df[col].iloc[i])
      if len(text_cell) > 0 and text_cell not in text_row:
        text_row += text_cell
    if len(text_row) > 0:
      merged_text.append(text_row)
    else:
      merged_text.append(np.nan)
  df["merged text"] = merged_text
  df = df.drop(columns = text_columns)
  return df

In [7]:
def map_target_var(award_price_divisions, y):
  y_encoded = []

  for i in range(len(y)):
    for j, interval in enumerate(award_price_divisions.keys()):
      lower_bound = award_price_divisions[interval][0]
      upper_bound = award_price_divisions[interval][1]

      if y[i] >= lower_bound and y[i] < upper_bound:
        y_encoded.append(j)
      else:
        continue

  #check for remaining values at the higher end of the array
  left_over_cases = len(y) - len(y_encoded)
  if left_over_cases > 0:
    for i in range(0, left_over_cases):
      y_encoded.append(j)

  y_encoded = np.array(y_encoded)

  return y_encoded

In [8]:
#we need to get a better feeling for the distribution accros award price "classes", is there a distribution to be found which could improve accuracy, propagating to the final model in MAE
def award_price_distribution(y, amount_of_classes: int, make_labels):
  if make_labels == True:
    award_price_divisions = {}
    y_sorted = np.sort(y)
    length_y = len(y_sorted)
    length_class = int(length_y / amount_of_classes)
    lower_index = 0
    for i in range(1, amount_of_classes):
      upper_index = length_class * i
      lower_value = y_sorted[lower_index]
      upper_value = y_sorted[upper_index]
      print("lower index: {}, lower value {}, upper index {}, upper value {}".format(lower_index, lower_value, upper_index, upper_value))
      award_price_divisions[str(lower_value) + "-" + str(upper_value)] = [lower_value, upper_value]
      print("length of class = {}".format(len(y_sorted[lower_index:upper_index])))
      lower_index += 1 * length_class

    y = map_target_var(award_price_divisions, y)
    return y
  else:
    print("no encoding or labels were made for the target variable")
    return y

In [10]:
def plot_metrics(ledger, save_path = None):
  #model results is a dict from a keras object
  model_results = ledger.history
  plt.figure(figsize=(10, 10))

  # Create the top row of subplots
  ax1 = plt.subplot(3, 1, 1)
  ax2 = plt.subplot(3, 1, 2)
  ax3 = plt.subplot(3, 1, 3)
  axes = [ax1, ax2, ax3]
  for i, key in enumerate(list(model_results.keys())[:int(0.5*len(model_results.keys()))]):
    loss_train = model_results[key]
    loss_val   = model_results["val_" + key]
    epochs = range(0,len(loss_train))

    axes[i].plot(epochs, loss_train, "g", label = "Training {}".format(key))
    axes[i].plot(epochs, loss_val, "b", label = "Validation {}".format("val_" + key))
    axes[i].title.set_text("Training and validation {}".format(key))
    axes[i].set_xlabel("Epochs")
    axes[i].set_ylabel("{}".format(key))
    axes[i].legend()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, format='png')

In [12]:
def train_validate_test_split(df, train, validate):
    target_col = "AWARD_VALUE_EURO_FIN_1"
    #df = df.sort_values(by = ["date_conclusion_contract"], axis = 0, ascending = True)
    train_indice = int(train * len(df))
    validate_indice = int(validate * len(df)) + train_indice
    print(train_indice, validate_indice)

    train_set = df[:train_indice]
    val_set = df[train_indice:validate_indice]
    test_set = df[validate_indice:]

    X_train = train_set.drop(columns = [target_col]).values
    y_train = train_set[target_col].values

    X_val = val_set.drop(columns = [target_col]).values
    y_val = val_set[target_col].values

    X_test = test_set.drop(columns = [target_col]).values
    y_test = test_set[target_col].values

    return X_train, y_train, X_val, y_val, X_test, y_test

In [13]:
#load the data
df = pd.read_pickle("/content/drive/MyDrive/Thesis/df_merged_dataset_clean").reset_index(drop = True)

In [ ]:
print(len(df_train), len(df_test))

In [ ]:
amount_of_classes = 10

X_train_num_cat, X_train_text, y_train, y_train_encoded = prepare_data(df_train,
                                                      upper_limit_outliers = 0.95,
                                                      lower_limit_outliers = 0.05,
                                                      train_or_test = "train",
                                                      amount_of_classes = amount_of_classes,
                                                      make_labels = True)

X_val_num_cat, X_test_num_cat, X_val_text, X_test_text, y_val, y_test, y_val_encoded, y_test_encoded = prepare_data(df_test,
                                                   train_or_test = "test",
                                                   amount_of_classes = amount_of_classes,
                                                   make_labels = True)

#y_train_label, y_train_encoding = map_target_var(y_train, "encoding")
#y_test_label, y_test_encoding = map_target_var(y_test, "encoding")

In [ ]:
type(y_train_encoded[0])

In [ ]:
word_count_list = []
for text in X_train_text:
  count = len(text.split())
  word_count_list.append(count)

df_word_count = pd.DataFrame(word_count_list)
df_word_count.describe()

In [ ]:
X_train_text[:5]

In [ ]:
subset_train = len(df_train) #or something else
subset_val = len(X_val_text)
subset_test = len(X_train_text)


#slice the target variable
y_train = y_train[:subset_train]
y_val = y_val[:subset_val]
y_test = y_test[:subset_test]

y_train_encoded = y_train_encoded[:subset_train]
y_val_encoded = y_val_encoded[:subset_val]
y_test_encoded = y_test_encoded[:subset_test]

#slice the text features
X_train_text = X_train_text[:subset_train]
X_val_text = X_val_text[:subset_val]
X_test_text = X_test_text[:subset_test]

#slice the num_cat features
X_train_num_cat = X_train_num_cat[:subset_train]
X_val_num_cat = X_val_num_cat[:subset_val]
X_test_num_cat = X_test_num_cat[:subset_test]


In [ ]:
y_train.shape

In [ ]:
transformer_model_name = "jplu/tf-xlm-roberta-base"
tokenizer = XLMRobertaTokenizer.from_pretrained(transformer_model_name)
max_length = 128

#Tokenize training set
train_input_ids = tokenizer(X_train_text, return_tensors="tf", max_length=max_length, padding="max_length", truncation=True)["input_ids"]
train_attention_mask = tokenizer(X_train_text, return_tensors="tf", max_length=max_length, padding="max_length", truncation=True)["attention_mask"]

#tokenize val set
val_input_ids = tokenizer(X_val_text, return_tensors="tf", max_length=max_length, padding="max_length", truncation=True)["input_ids"]
val_attention_mask = tokenizer(X_val_text, return_tensors="tf", max_length=max_length, padding="max_length", truncation=True)["attention_mask"]

#tokenize test set
test_input_ids = tokenizer(X_test_text, return_tensors="tf", max_length=max_length, padding="max_length", truncation=True)["input_ids"]
test_attention_mask = tokenizer(X_test_text, return_tensors="tf", max_length=max_length, padding="max_length", truncation=True)["attention_mask"]

-----RUN THE HUGGINGFACE MODEL WITH ONLY MULTIPLE HEADS BASED ON THE ENCODING OF THE TARGET VARIABLE-----

In [ ]:
amount_of_targets = amount_of_classes - 1

#define path for saving model
checkpoint_path = "/content/drive/MyDrive/Thesis/checkpoint.ckpt"
checkpoint_dir = os.path.dirname(checkpoint_path)

# Create a callback that saves the model's weights
cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_path,save_weights_only=True,verbose=1, save_freq=2*32)

#train the model directly on the classification task
transformer_model_name = "jplu/tf-xlm-roberta-base"
language_model = TFAutoModelForSequenceClassification.from_pretrained(transformer_model_name, num_labels=amount_of_targets)

#specify model parameters
batch_size = 32
num_epochs = 3

num_train_steps = len(y_train) / batch_size * num_epochs
lr_scheduler = PolynomialDecay(
    initial_learning_rate=5e-5, end_learning_rate=0.0, decay_steps=num_train_steps
)

optimizer = Adam(learning_rate=lr_scheduler)
loss = losses.SparseCategoricalCrossentropy(from_logits=True) #the model outputs logits rather than probabilities hence passing from_logits=True
metrics = tf.metrics.SparseCategoricalAccuracy()

#compile the model
language_model.compile(optimizer=optimizer, loss = loss, metrics = metrics)

#fit the model
history_transformer = language_model.fit(x = [train_input_ids, train_attention_mask], y=y_train_encoded,
                                        validation_data=([val_input_ids, val_attention_mask], y_val_encoded),
                                        epochs=num_epochs,
                                        batch_size = batch_size,
                                        callbacks = [cp_callback])

-----RUN THE COMBINED MODEL-----

In [ ]:
#load the pretrained models and combine them
num_cat_model = tf.keras.models.load_model("/content/drive/MyDrive/Thesis/model_num_cat.keras")


In [ ]:
num_cat_model.summary()
num_cat_model.layers[-2].output

In [ ]:
#DEFINE TRANSFORMER MODEL
amount_of_targets = amount_of_classes - 1
transformer_model_name = "jplu/tf-xlm-roberta-base"
transformer = TFAutoModelForSequenceClassification.from_pretrained(transformer_model_name, num_labels=amount_of_targets)

#input_ids = tf.keras.Input(shape=(max_length,), dtype='int32')
#attention_mask = tf.keras.Input(shape=(max_length, ), dtype='int32')
outputs = transformer(train_input_ids, train_attention_mask).logits

In [ ]:
#outputs is expected to be logits of length 9 as there are 9 classes
outputs

In [ ]:
#FEED LOGITS TO THE SOFTMAX TO ASSESS THE OUTPUT
softmax_output = tf.nn.softmax(outputs)
softmax_output[0]

In [ ]:
#let's play around a little more with the keras model
loss_function = "mae"
metrics = ["mse"]
epochs = 1
batch_size = 32
#define the layers
input_num_cat = Input(shape=X_train_num_cat.shape[1])
x = Dense(256, activation = "relu")(input_num_cat) #kernel_regularizer=regularizers.L1L2(l1=0.005)
x = Dropout(rate = 0.2)(x)
x = Dense(128, activation = "relu")(x)
x = Dropout(rate=0.1)(x)
x = Dense(32, activation = "relu")(x)
x = Dropout(rate=0.1)(x)
x = Dense(64, activation = "relu")(x)
x = Dropout(rate=0.1)(x)
x = Dense(32, activation = "relu")(x)
regression_layer = Dense(1, activation="linear")(x)
model_num_cat = Model(inputs = [input_num_cat],
                      outputs = regression_layer)

num_train_steps = len(X_train_num_cat) / batch_size * epochs
lr_scheduler = PolynomialDecay(
    initial_learning_rate=0.05, end_learning_rate=0.00001, decay_steps=num_train_steps
)

checkpoint_path = "/content/drive/MyDrive/Thesis"
cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_path, verbose=1)

optimizer = Adam(learning_rate=lr_scheduler)
num_train_steps

model_num_cat.compile(loss=loss_function,
                      optimizer = optimizer,
                      metrics = metrics)
model_num_cat.summary()

history = model_num_cat.fit(x = [X_train_num_cat], y=y_train,
                              validation_data = (X_val_num_cat, y_val),
                              epochs = epochs,
                              batch_size = batch_size,
                              callbacks=[cp_callback])

y_pred = model_num_cat.predict(X_test_num_cat)
mae = mean_absolute_error(y_test, y_pred)

metric = tf.keras.metrics.R2Score()
metric.update_state(y_test.reshape(-1,1), y_pred)
r2_result = metric.result()
r2_result.numpy()

#model_num_cat, history, mae, r2_result

In [ ]:
#with open("/content/drive/MyDrive/Thesis/history_test", "wb") as f:
#  pickle.dump(history, f)

In [ ]:
with open("/content/drive/MyDrive/Thesis/history_test", "rb") as f:
  history = pickle.load(f)

In [ ]:
from tensorflow.keras.utils import plot_model
keras.utils.plot_model(num_cat_model, show_shapes=True)

In [ ]:
#SAVE THE FINAL VERSION OF THE MODEL
model_num_cat.save("/content/drive/MyDrive/Thesis/test_model.keras")

In [ ]:
#LOAD THE FINAL VERSION OF THE MODEL
num_cat_model = tf.keras.models.load_model("/content/drive/MyDrive/Thesis/test_model.keras")

In [ ]:
amount_of_classes = 10

X_train_num_cat, X_train_text, y_train, y_train_encoded = prepare_data(df_train,
                                                      upper_limit_outliers = 0.95,
                                                      lower_limit_outliers = 0.05,
                                                      train_or_test = "train",
                                                      amount_of_classes = amount_of_classes,
                                                      make_labels = True)

X_val_num_cat, X_test_num_cat, X_val_text, X_test_text, y_val, y_test, y_val_encoded, y_test_encoded = prepare_data(df_test,
                                                   train_or_test = "test",
                                                   amount_of_classes = amount_of_classes,
                                                   make_labels = True)

In [ ]:
transformer_model_name = "jplu/tf-xlm-roberta-base"
tokenizer = XLMRobertaTokenizer.from_pretrained(transformer_model_name)
max_length = 128

#Tokenize training set
train_input_ids = tokenizer(X_train_text, return_tensors="tf", max_length=max_length, padding="max_length", truncation=True)["input_ids"]
train_attention_mask = tokenizer(X_train_text, return_tensors="tf", max_length=max_length, padding="max_length", truncation=True)["attention_mask"]

#tokenize val set
val_input_ids = tokenizer(X_val_text, return_tensors="tf", max_length=max_length, padding="max_length", truncation=True)["input_ids"]
val_attention_mask = tokenizer(X_val_text, return_tensors="tf", max_length=max_length, padding="max_length", truncation=True)["attention_mask"]

#tokenize test set
test_input_ids = tokenizer(X_test_text, return_tensors="tf", max_length=max_length, padding="max_length", truncation=True)["input_ids"]
test_attention_mask = tokenizer(X_test_text, return_tensors="tf", max_length=max_length, padding="max_length", truncation=True)["attention_mask"]

In [ ]:
#DEFINE VARIABLES
input_dim_num_cat=X_train_num_cat.shape[1]
max_length = 128
batch_size = 32
epochs = 5

#LOAD PRETRAINED NUM_CAT_MODEL
num_cat_model = tf.keras.models.load_model("/content/drive/MyDrive/Thesis/test_model.keras")

#INSTANTIATE NEW MODEL WITHOUT PREDICTION HEAD
c_num_cat_model = Model(inputs=num_cat_model.inputs, outputs=num_cat_model.layers[-2].output)

#MAKE THE FINAL MODEL
input_num_cat                     = Input(shape=input_dim_num_cat, name="numerical and categorical input")
output_num_cat                    = c_num_cat_model(input_num_cat)
batch_normalization_num_cat       = keras.layers.BatchNormalization()(output_num_cat)

#INITIALIZE THE TRANSFORMER MODEL
amount_of_targets = amount_of_classes - 1
transformer_model_name = "jplu/tf-xlm-roberta-base"
transformer = TFAutoModelForSequenceClassification.from_pretrained(transformer_model_name, num_labels=amount_of_targets)

#DEFINE INPUTS AND FEED TO THE TRANSFORMER MODEL
input_ids = tf.keras.Input(shape=(max_length,), dtype='int32', name="inputs_ids")
attention_mask = tf.keras.Input(shape=(max_length,), dtype='int32', name = "attention_mask")
transformer_logits = transformer(input_ids, attention_mask).logits
batch_normalization_transformer = keras.layers.BatchNormalization()(transformer_logits)

#COMBINE THE MODEL OUTPUTS
concatentate = concatenate([batch_normalization_num_cat, batch_normalization_transformer], name="concatenation_layer")
final_layer = Dense(1, activation = "linear", name="regression_layer")(concatentate)

#create final model
final_model = Model(inputs=[{"input_ids": input_ids, "attention_mask": attention_mask}, input_num_cat],
                    outputs=[final_layer],
                    name="merged")

#DEFINE LEARNING RATE DECAY FOR OPTIMIZATION
num_train_steps = len(y_train) / batch_size * epochs
lr_scheduler = PolynomialDecay(initial_learning_rate=5e-5,
                               end_learning_rate=0.0,
                               decay_steps=num_train_steps)

#DEFINE CUSTOM METRIC
def coeff_determination(y_true, y_pred):
    SS_res =  K.sum(K.square( y_true-y_pred ))
    SS_tot = K.sum(K.square( y_true - K.mean(y_true) ) )
    return ( 1 - SS_res/(SS_tot + K.epsilon()) )

#define custom loss function for combining classification with regression
loss_function = tf.keras.losses.MeanAbsoluteError(reduction="auto", name="mean_absolute_error")
optimizer = tf.keras.optimizers.Adam(learning_rate=lr_scheduler)
metrics = ["mse", coeff_determination]

final_model.compile(loss=loss_function,
                   optimizer=optimizer,
                    metrics=metrics)

In [ ]:
#DEFINE CALLBACKS TO SAVE THE MODEL DURING RUNTIME
checkpoint_path = "/content/drive/MyDrive/Thesis/results_combined_model/"
n_batches = int(len(X_train_num_cat) / batch_size)

cp_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    verbose=1,
    save_freq=n_batches)

history_final_model = final_model.fit(x=[[train_input_ids, train_attention_mask], X_train_num_cat], y=y_train,
                validation_data=([[val_input_ids, val_attention_mask], X_val_num_cat], y_val),
                epochs=5,
                batch_size = 32,
                callbacks=[cp_callback])

In [ ]:
with open("/content/drive/MyDrive/Thesis/history_final_model", "wb") as f:
  pickle.dump(history_final_model, f)

In [ ]:
from google.colab import files

In [ ]:
files.download("/content/drive/MyDrive/Thesis/history_final_model")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
final_model.summary()

In [ ]:
plot_metrics(ledger=history_final_model)

In [ ]:
from tensorflow.keras.utils import plot_model
keras.utils.plot_model(final_model, show_shapes=True)

In [ ]:
amount_of_targets = 6
max_length = 128

#define the numeric and categorical model
input_dim_num_cat = 55 #len(X_train[0])
input_num_cat = Input(shape=input_dim_num_cat)
num_cat_layer = Dense(68, activation = "relu", kernel_regularizer=regularizers.L1L2(l1=0.001))(input_num_cat)
dropout_layer = Dropout(rate=0.3)(num_cat_layer)
num_cat_layer = Dense(32, activation="relu", kernel_regularizer=regularizers.L1L2(l1=0.001))(dropout_layer)
num_cat_layer_final = Dense(8, activation="relu")(num_cat_layer)

input_ids = tf.keras.Input(shape=(max_length,), dtype='int32')
attention_mask = tf.keras.Input(shape=(max_length, ), dtype='int32')
encoding_transformer = transformer(input_ids, attention_mask)
logits = encoding_transformer[0]
probabilities = Dense(amount_of_targets, activation="softmax")(logits)

#concat layers
combined_input = concatenate([probabilities, num_cat_layer_final], name="concatenated_layer")
final_layer = Dense(1, activation="linear", name="output_layer")(combined_input)

#create final model
final_model = Model(inputs=[{"input_ids": input_ids, "attention_mask": attention_mask}, input_num_cat],
                    outputs=[final_layer],
                    name="merged")

#define custom loss function for combining classification with regression
classification_loss = losses.SparseCategoricalCrossentropy(from_logits=False)
loss_function = "mean_absolute_error"
optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)
metrics = ["mae", "mse"]

final_model.compile(loss=loss_function,
                   optimizer=optimizer,
                    metrics=metrics)

In [ ]:
from tensorflow.keras.utils import plot_model
keras.utils.plot_model(final_model, show_shapes=True)

In [ ]:
final_model.fit(x=[[train_input_ids, train_attention_mask], X_train_num_cat], y=y_train,
                  validation_data=([[test_input_ids, test_attention_mask], X_test_num_cat], y_test),
                  epochs=5, batch_size = 32)

THE MODEL BELOW IS THE OLD VERSION (PROBABLY THE RIGHT ONE TOO BUT ALRIGHT)

In [ ]:
#now let's make a model for our categorical and numerical data
#define variables
optimizer = keras.optimizers.Adam(learning_rate=0.01) #tf.keras.optimizers.experimental.Adagrad(learning_rate=0.001)
loss_function = "mean_absolute_error"

#define the numeric and categorical model
input_dim_num_cat = 55 #len(X_train[0])
input_num_cat = Input(shape=input_dim_num_cat)
num_cat_layer_1 = Dense(128, activation = "relu")(input_num_cat)
num_cat_layer_2 = Dense(64, activation="relu")(num_cat_layer_1)
num_cat_layer_3 = Dense(32, activation="relu")(num_cat_layer_2)
num_cat_layer_4 = Dense(10, activation="relu")(num_cat_layer_3)

model_name = "jplu/tf-xlm-roberta-base"
num_of_classes = 10

input_ids = tf.keras.Input(shape=(max_length,), dtype='int32')
attention_mask = tf.keras.Input(shape=(max_length, ), dtype='int32')
transformer = TFAutoModelForSequenceClassification.from_pretrained(model_name, num_labels=10)
encoded = transformer({"input_ids": input_ids, "attention_mask": attention_mask})
logits = encoded[0]
probabilities = Dense(num_of_classes, activation="softmax")(logits)
#model = tf.keras.models.Model(inputs = {"input_ids": input_ids, "attention_mask": attention_mask}, outputs = probabilities)

combined_input = concatenate([probabilities, num_cat_layer_4], name="concatenated_layer")
final_layer = Dense(1, activation="linear", name="output_layer")(combined_input)
final_model = Model(inputs=[{"input_ids": input_ids, "attention_mask": attention_mask}, input_num_cat],
                    outputs=[final_layer],
                    name="merged")

#define custom loss function for combining classification with regression
classification_loss = losses.SparseCategoricalCrossentropy(from_logits=False)
loss_function = "mean_absolute_error"
optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)

final_model.compile(loss=loss_function,
                   optimizer=optimizer)

In [ ]:
from tensorflow.keras.utils import plot_model
keras.utils.plot_model(final_model, show_shapes=True)

In [ ]:
final_model.fit(x=[[train_input_ids, train_attention_mask], X_train], y=y_train,
                  validation_data=([[test_input_ids, test_attention_mask], X_test], y_test),
                  epochs=5, batch_size = 32)

In [ ]:
#reshape data for randomforestregressor
y_train = np.ravel(y_train)
y_test = np.ravel(y_test)

In [ ]:
#let's fit a randomforrest on the data to see if this provides better results
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import make_scorer, mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV
random_forrest_regressor = RandomForestRegressor(n_estimators=100, random_state=0)
random_forrest_regressor.fit(X_train, y_train)
y_pred = random_forrest_regressor.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

In [ ]:
print("r2: {} ""\n\n"", MAE: {}".format(r2, mae))

In [ ]:
# Define the parameter grid for the grid search
param_grid = {
    'n_estimators': [50, 100, 150, 300],  # You can add more values to search over
    'max_depth': [None, 10, 20],      # You can add more values to search over
    'min_samples_split': [2, 5, 10],  # You can add more values to search over
    "criterion": [“squared_error”, “absolute_error”]
}

# Define the scoring metrics for GridSearchCV
scoring = {
    'r2': make_scorer(r2_score),
    'mae': make_scorer(mean_absolute_error),
}

In [ ]:
y_train